# 🗄️ Notebook 3: Database-Backed Deduplication (Exactly-Once Semantics)

An in-memory map vanishes on restart and isn't shared across replicas. For real *exactly-once* semantics, store the idempotency key inside the **same transaction** that performs the side effect, with a `UNIQUE` constraint to enforce uniqueness across all callers.


## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 SQLite implementation (no external services)

In [ ]:
import sqlite3, uuid, json, threading

conn = sqlite3.connect(':memory:', check_same_thread=False)
conn.executescript('''
CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER);
CREATE TABLE idem (key TEXT PRIMARY KEY, result_json TEXT NOT NULL);
INSERT INTO accounts VALUES ('alice', 100);
''')
lock = threading.Lock()

def charge(key, account, amount):
    with lock, conn:  # 'with conn' = transaction
        row = conn.execute('SELECT result_json FROM idem WHERE key=?', (key,)).fetchone()
        if row:
            return ('REPLAY', json.loads(row[0]))
        conn.execute('UPDATE accounts SET balance = balance - ? WHERE name=?', (amount, account))
        bal = conn.execute('SELECT balance FROM accounts WHERE name=?', (account,)).fetchone()[0]
        result = {'balance': bal, 'charged': amount}
        try:
            conn.execute('INSERT INTO idem(key, result_json) VALUES (?, ?)', (key, json.dumps(result)))
        except sqlite3.IntegrityError:
            # A concurrent request inserted first — abort and retry as replay
            raise
        return ('FRESH', result)

k = str(uuid.uuid4())
for i in range(3):
    print(f'attempt {i}:', charge(k, 'alice', 10))
print('balance:', conn.execute('SELECT balance FROM accounts WHERE name="alice"').fetchone()[0])


## 🧠 Why a transaction?

Putting the *side effect* and the *idempotency key insert* in the **same transaction** means either both happen or neither does. Without it, you can crash between the two and either:

- charge twice (key not inserted yet), or
- never charge (key inserted but balance update rolled back).

## 🧹 Operational notes

- Idempotency tables grow forever; expire keys after the retry window (e.g. 24h).
- Put the key in a header (`Idempotency-Key`) so middleware can enforce it uniformly.
- Never reuse an idempotency key for a *different* request body — Stripe rejects this with a 400.